# CantioAI Quickstart\n\nThis notebook demonstrates the basic usage of CantioAI for voice conversion.

In [ ]:
# Import necessary libraries\nimport torch\nimport numpy as np\nimport matplotlib.pyplot as plt\n%matplotlib inline\n\n# Import CantioAI components\nfrom cantioai.src.models.hybrid_svc import HybridSVC\nfrom cantioai.src.data.preprocess import WorldFeatureExtractor\nfrom cantioai.src.inference.synthesizer import VocoderInference\nfrom cantioai.src.utils.logging import setup_logging\n\n# Setup logging\nsetup_logging()\n

## 1. Initialize Model\n\nCreate a HybridSVC model with default parameters.

In [ ]:
# Initialize model\nmodel = HybridSVC(\n    phoneme_feature_dim=32,\n    spectral_envelope_dim=60,\n    speaker_embed_dim=128,\n    n_speakers=10,\n    use_pitch_quantizer=True\n)\n\nprint(f"Model: {model}")\nprint(f"Number of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Feature Extraction (Example)\n\nDemonstrate how to extract features from audio using WORLD vocoder.

In [ ]:
# Initialize feature extractor\nextractor = WorldFeatureExtractor(\n    sample_rate=24000,\n    frame_period=5.0,\n    fft_size=1024,\n    num_mcep=60\n)\n\n# Create dummy audio signal (replace with real audio loading)\nsr = 24000\nduration = 1.0\nt = np.linspace(0, duration, int(sr * duration))\nfrequency = 220.0\nwaveform = 0.5 * np.sin(2 * np.pi * frequency * t)\n\n# Extract features\nfeatures = extractor.extract_features(waveform)\n\nprint("Extracted features:")\nfor key, value in features.items():\n    if isinstance(value, np.ndarray):\n        print(f"  {key}: {value.shape} ({value.dtype})")\n    else:\n        print(f"  {key}: {value}")

## 3. Inference Example\n\nRun inference with the model on dummy data.

In [ ]:
# Create dummy inputs\nbatch_size = 1\nseq_len = 50\nphoneme_features = torch.randn(batch_size, seq_len, 32)\nf0_hz = torch.rand(batch_size, seq_len, 1) * 200 + 100\nspk_id = torch.randint(0, 10, (batch_size,))\n\n# Set model to eval mode\nmodel.eval()\n\n# Forward pass\nwith torch.no_grad():\n    sp_pred, f0_quant, extras = model(\n        phoneme_features=phoneme_features,\n        f0=f0_hz,\n        spk_id=spk_id,\n        f0_is_hz=True,\n        return_quantized_f0=True\n    )\n\nprint(f"Input shapes:")\nprint(f"  phoneme_features: {phoneme_features.shape}")\nprint(f"  f0_hz: {f0_hz.shape}")\nprint(f"  spk_id: {spk_id.shape}")\nprint(f"\nOutput shapes:")\nprint(f"  sp_pred: {sp_pred.shape}")\nprint(f"  f0_quant: {f0_quant.shape if f0_quant is not None else None}")\nprint(f"  extras keys: {list(extras.keys()) if extras else []}")

## 4. Audio Synthesis Example\n\nDemonstrate how to synthesize audio from predicted features.

In [ ]:
# Import WORLD vocoder\nfrom cantioai.src.inference.vocoder import WORLDVocoder\n\n# Initialize vocoder\nvocoder = WORLDVocoder(sample_rate=24000, frame_period=5.0)\n\n# Convert predicted spectral envelope to world coded form\nsp_world = vocoder.decode_spectral_envelope(\n    sp_pred.squeeze(0).cpu().numpy(),\n    fs=24000,\n    fft_size=1024\n)\n\n# Synthesize audio\nwaveform = vocoder.synthesize(\n    f0=f0_hz.squeeze(0).cpu().numpy(),\n    coded_sp=sp_world,\n    ap=np.zeros_like(sp_world)  # Simple voiced aperiodicity\n)\n\nprint(f"Synthesized waveform shape: {waveform.shape}")\nprint(f"Waveform dtype: {waveform.dtype}")\nprint(f"Waveform range: [{waveform.min():.3f}, {waveform.max():.3f}]")

## 5. Next Steps\n\n- Train the model on your dataset using `scripts/train.py`\n- Extract features from audio using `scripts/preprocess.py`\n- Run inference using `scripts/infer.py`\n- Evaluate model performance using `scripts/evaluate.py`\n- Explore the source code for customization options